In [ ]:
# %pip install tableauhyperapi
# %pip install tableauserverclient

/Users/ruobin.chang/Documents/Develop/tableau-hyper-extract-data/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.
/Users/ruobin.chang/Documents/Develop/tableau-hyper-extract-data/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


## 1. Downloading tdsx files via the Tableau Server Client


In [5]:
## Set the required parameters
import os

from dotenv import load_dotenv

_ = load_dotenv()

SERVER_NAME = os.getenv("SERVER_NAME")
SITE_NAME = os.getenv("SITE_NAME")
PAT_NAME = os.getenv("PAT_NAME")
PAT_VALUE = os.getenv("PAT_VALUE")
DATASOURCE_ID = os.getenv("DATASOURCE_ID")
OUTPUT_PATH = "./"

In [6]:
import tableauserverclient as TSC

# Authentication (PAT)
tableau_auth = TSC.PersonalAccessTokenAuth(PAT_NAME, PAT_VALUE, site_id=SITE_NAME)
server = TSC.Server(SERVER_NAME, use_server_version=True)

with server.auth.sign_in(tableau_auth):
    # Data source download (including extract)
    file_path = server.datasources.download(
        DATASOURCE_ID,
        filepath=OUTPUT_PATH,
        include_extract=True,
    )
    print(f"Downloaded to: {file_path}")

Downloaded to: /Users/ruobin.chang/Documents/Develop/tableau-hyper-extract-data/HyperTest.tdsx


## 2. Extract Hyper from .tdsx


In [7]:
import os
import zipfile

with zipfile.ZipFile(file_path, "r") as z:
    for name in z.namelist():
        if name.endswith(".hyper"):
            z.extract(name, "./")
            hyper_path = os.path.join("./", name)
            print(f"Extracted: {hyper_path}")


Extracted: ./Data/Extracts/federated_1nek68b0zlyn201endnbk0.hyper


## 3. Extracting data via the Hyper API


In [8]:
import shutil
from pathlib import Path

from tableauhyperapi import (
    Connection,
    HyperProcess,
    TableName,
    Telemetry,
)


def run_read_data_from_existing_hyper_file():
    """
    An example of how to read and print data from an existing Hyper file.
    """
    print("EXAMPLE - Read data from an existing Hyper file")

    # Path to a Hyper file containing all data inserted into Customer, Product, Orders and LineItems table.
    # See "insert_data_into_multiple_tables.py" for an example that works with the complete schema.
    path_to_source_database = hyper_path

    # Make a copy of the superstore denormalized sample Hyper file
    path_to_database = Path(
        shutil.copy(
            src=path_to_source_database, dst="superstore_sample_denormalized_read.hyper"
        )
    ).resolve()

    # Starts the Hyper Process with telemetry enabled to send data to Tableau.
    # To opt out, simply set telemetry=Telemetry.DO_NOT_SEND_USAGE_DATA_TO_TABLEAU.
    with HyperProcess(telemetry=Telemetry.SEND_USAGE_DATA_TO_TABLEAU) as hyper:
        # Connect to existing Hyper file "superstore_sample_denormalized_read.hyper".
        with Connection(
            endpoint=hyper.endpoint, database=path_to_database
        ) as connection:
            # The table names in the "Extract" schema (the default schema).
            table_names = connection.catalog.get_table_names(schema="Extract")

            for table in table_names:
                table_definition = connection.catalog.get_table_definition(name=table)
                print(f"Table {table.name} has qualified name: {table}")
                for column in table_definition.columns:
                    print(
                        f"Column {column.name} has type={column.type} and nullability={column.nullability}"
                    )
                print("")

            # Print all rows from the "Extract"."Extract" table.
            table_name = TableName(table_names[-1].schema_name, table_names[-1].name)
            print(f"These are all rows in the table {table_name}:")
            # `execute_list_query` executes a SQL query and returns the result as list of rows of data,
            # each represented by a list of objects.
            rows_in_table = connection.execute_list_query(
                query=f"SELECT * FROM {table_name}"
            )
            print(rows_in_table)

        print("The connection to the Hyper file has been closed.")
    print("The Hyper process has been shut down.")


run_read_data_from_existing_hyper_file()

EXAMPLE - Read data from an existing Hyper file
Table "返品_E3889AE223A64AEA9660763A5917E711" has qualified name: "Extract"."返品_E3889AE223A64AEA9660763A5917E711"
Column "オーダー ID" has type=TEXT and nullability=Nullability.NULLABLE
Column "返品" has type=TEXT and nullability=Nullability.NULLABLE

Table "関係者_D6DAAC58E82146128B5081F9B5F35AE9" has qualified name: "Extract"."関係者_D6DAAC58E82146128B5081F9B5F35AE9"
Column "地域" has type=TEXT and nullability=Nullability.NULLABLE
Column "地域マネージャー" has type=TEXT and nullability=Nullability.NULLABLE

Table "オーダー_AE1D8F2670F44AB5AD340E1B9AF68ADB" has qualified name: "Extract"."オーダー_AE1D8F2670F44AB5AD340E1B9AF68ADB"
Column "行 ID" has type=BIG_INT and nullability=Nullability.NULLABLE
Column "オーダー ID" has type=TEXT and nullability=Nullability.NULLABLE
Column "オーダー日" has type=DATE and nullability=Nullability.NULLABLE
Column "出荷日" has type=DATE and nullability=Nullability.NULLABLE
Column "出荷モード" has type=TEXT and nullability=Nullability.NULLABLE
Column "顧客 ID

## 參考Docs

- https://github.com/tableau/hyper-api-samples/blob/main/Tableau-Supported/Python/read_and_print_data_from_existing_hyper_file.py

- https://tableau.github.io/hyper-db/docs/guides/hyper_file/read
